# ⚡ 使用 GitHub 模型的並行代理工作流程（Python）

## 📋 高階並行處理教學

此筆記本展示了使用 Microsoft Agent Framework 的<strong>並行工作流程範例</strong>。你將學會如何構建高效能的並行處理工作流程，在其中多個 AI 代理同時執行，大幅提升吞吐量並實現複雜的多線程商業流程。

## 🎯 學習目標

### 🚀 <strong>並行處理基礎</strong>
- <strong>平行代理執行</strong>：同時運行多個代理以達到最高效率
- <strong>工作流程協調</strong>：協調並行操作同時保持資料一致性
- <strong>效能優化</strong>：透過並行處理實現顯著的速度提升
- <strong>資源管理</strong>：有效利用 AI 模型資源於並發操作中

### 🏗️ <strong>進階並行模式</strong>
- **分叉-合併處理**：將工作分配給多個代理並合併結果
- <strong>流水線並行</strong>：疊代執行階段以持續提高吞吐量
- <strong>負載均衡</strong>：在可用代理資源間平均分配工作
- <strong>同步點</strong>：在關鍵工作流程階段協調並行代理

### 🏢 <strong>企業級並行應用</strong>
- <strong>大量文件處理</strong>：同時處理多份文件
- <strong>即時內容分析</strong>：併行分析即時輸入資料流
- <strong>批次處理優化</strong>：最大化大規模作業的吞吐量
- <strong>多模態分析</strong>：並行處理不同內容類型（文字、圖像、數據）

## ⚙️ 先決條件與設定

### 📦 <strong>必要依賴</strong>

安裝具備並行工作流程功能的 Agent Framework：

```bash
pip install agent-framework-core -U
```

### 🔑 **GitHub 模型設定**

**環境設定 (.env 檔案)：**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

**並行處理注意事項：**
- <strong>速率限制</strong>：監控 GitHub 模型 API 的並行請求速率限制
- <strong>資源使用</strong>：考量多代理並行時的記憶體與 CPU 使用量
- <strong>錯誤處理</strong>：為平行操作實作健全的錯誤回復機制

### 🏗️ <strong>並行工作流程架構</strong>

```mermaid
graph TD
    A[工作流程開始] --> B[並行執行]
    B --> C[代理池 1]
    B --> D[代理池 2]
    B --> E[代理池 3]
    C --> F[結果彙總]
    D --> F
    E --> F
    F --> G[最終輸出]
    
    H[GitHub 模型 API] --> C
    H --> D
    H --> E
```

**主要優點：**
- **⚡ 效能**：透過平行執行獲得顯著速度提升
- **📈 可擴展性**：處理增加的工作量時，耗時不成比例增加
- **🔄 效率**：更有效利用現有計算資源
- **🎯 吞吐量**：在相同時間內處理更多工作

## 🎨 <strong>並行工作流程設計模式</strong>

### 🔍 <strong>研究與分析流程</strong>
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 <strong>資料處理工作流程</strong>
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 <strong>內容創建流程</strong>
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 <strong>多階段處理</strong>
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 <strong>企業效能收益</strong>

### ⚡ <strong>吞吐量優化</strong>
- <strong>並行執行</strong>：多個代理同時工作
- <strong>資源利用</strong>：最大化可用 AI 模型容量效率
- <strong>時間縮短</strong>：大幅降低總處理時間
- <strong>可擴展架構</strong>：依需求輕鬆新增更多並行代理

### 🛡️ <strong>可靠性與彈性</strong>
- <strong>容錯能力</strong>：單一代理失敗不會終止整體工作流程
- <strong>錯誤隔離</strong>：一個並行分支的問題不影響其他分支
- <strong>優雅降級</strong>：即使代理能力減少系統仍能持續運作
- <strong>復原機制</strong>：失敗作業自動重試與錯誤處理

### 📊 <strong>監控與可觀測性</strong>
- <strong>並行執行追蹤</strong>：監控所有同時操作的進度
- <strong>效能指標</strong>：衡量速度與效率提升
- <strong>資源使用分析</strong>：優化並行代理配置
- <strong>瓶頸識別</strong>：找出並解決效能限制

讓我們一起打造高效能並行 AI 工作流程！🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = await provider.create_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = await provider.create_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)

In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
